In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv("/kaggle/input/notebooks/shreyaashambhavii/exploratory-data-analysis-and-preprocessing/train.csv")
test = pd.read_csv("/kaggle/input/notebooks/shreyaashambhavii/exploratory-data-analysis-and-preprocessing/test.csv")

In [3]:
train

,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService_DSL,InternetService_FiberOptic,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod_ElectronicCheck,PaymentMethod_MailedCheck,PaymentMethod_CreditCard,PaymentMethod_BankTransfer,MonthlyCharges,TotalCharges,Churn
0,0,0,1,1,3,1,0,1,0,1,...,0,1,1,0,1,0,0,2,3,0
1,0,0,1,1,5,1,0,1,0,1,...,0,2,0,0,0,1,0,2,3,0
2,0,0,1,0,5,1,1,0,1,0,...,1,0,1,1,0,0,0,4,4,0
3,1,0,0,0,1,1,0,0,1,0,...,0,0,1,1,0,0,0,2,1,1
4,1,0,0,0,1,1,0,0,1,0,...,0,0,1,1,0,0,0,2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,0,0,0,0,5,1,1,0,1,0,...,1,2,0,0,0,0,1,4,4,0
594190,1,0,0,0,6,1,1,1,0,1,...,1,2,0,0,0,0,1,4,4,0
594191,1,0,1,0,6,1,1,0,0,0,...,0,2,0,0,0,1,0,1,3,0
594192,1,0,0,0,3,1,1,0,1,0,...,1,0,1,1,0,0,0,3,3,0


In [4]:
test

,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService_DSL,InternetService_FiberOptic,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod_ElectronicCheck,PaymentMethod_MailedCheck,PaymentMethod_CreditCard,PaymentMethod_BankTransfer,MonthlyCharges,TotalCharges
0,1,0,1,0,6,1,1,0,1,1,...,1,1,2,1,1,0,0,0,4,4
1,1,0,1,0,6,1,0,0,0,0,...,0,0,2,0,0,0,0,1,1,2
2,0,0,0,0,1,1,0,1,0,1,...,0,0,0,1,0,0,0,1,2,1
3,0,0,1,1,6,1,1,1,0,1,...,1,1,2,0,0,0,1,0,3,4
4,1,0,0,0,2,1,0,0,1,1,...,1,1,0,0,1,0,0,0,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254650,0,0,1,1,6,1,0,0,0,0,...,0,0,2,0,0,0,1,0,1,3
254651,0,1,1,0,2,1,1,0,1,0,...,1,1,0,1,1,0,0,0,4,3
254652,0,0,1,0,3,1,1,0,1,1,...,1,1,0,1,0,0,0,1,4,3
254653,1,0,0,0,3,1,0,0,0,0,...,0,0,2,1,0,0,1,0,1,1


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier

import joblib
from scipy.stats import randint, uniform

In [6]:
import warnings
warnings.filterwarnings("ignore")

In [7]:
X = train.drop('Churn', axis = 1)
y = train['Churn']

In [8]:
results = {}
best_estimators = {}

In [9]:
print("--- Logistic Regression ---")

logreg_pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter = 1000, random_state = 42, class_weight = "balanced"))])

logreg_param_grid = {"clf__C": [0.01, 0.1, 1, 10, 100],
                     "clf__penalty": ["l1", "l2"],
                     }

logreg_model = GridSearchCV(logreg_pipe, logreg_param_grid, cv = 5, scoring = "roc_auc", n_jobs = -1)
logreg_model.fit(X, y)

print(f"Best ROC AUC: {logreg_model.best_score_:.4f}")
print(f"Best Params: {logreg_model.best_params_}")

--- Logistic Regression ---
Best ROC AUC: 0.9063
Best Params: {'clf__C': 1, 'clf__penalty': 'l2'}


In [10]:
results["Logistic Regression"] = logreg_model.best_score_
best_estimators["Logistic Regression"] = logreg_model.best_estimator_

In [11]:
print("--- Random Forest ---")

rf_pipe = Pipeline([("clf", RandomForestClassifier(class_weight = "balanced", random_state = 42, n_jobs = 1))])

rf_param_grid = {"clf__n_estimators": randint(100, 500),
                 "clf__max_depth": [None, 10, 20, 30],
                 "clf__min_samples_split": randint(2, 20),
                 "clf__min_samples_leaf": randint(1, 10),
                 "clf__max_features": ["sqrt", "log2", 0.3]}

rf_model = RandomizedSearchCV(rf_pipe, rf_param_grid, n_iter = 10, cv = 5, scoring = "roc_auc", n_jobs = -1, verbose = 1)
rf_model.fit(X, y)

print(f"Best ROC AUC: {rf_model.best_score_:.4f}")
print(f"Best Params: {rf_model.best_params_}")

--- Random Forest ---
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best ROC AUC: 0.9076
Best Params: {'clf__max_depth': 10, 'clf__max_features': 0.3, 'clf__min_samples_leaf': 8, 'clf__min_samples_split': 18, 'clf__n_estimators': 360}


In [12]:
results["Random Forest"] = rf_model.best_score_
best_estimators["Random Forest"] = rf_model.best_estimator_

In [13]:
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [14]:
neg, pos         = (y == 0).sum(), (y == 1).sum()
scale_pos_weight = neg / pos

In [15]:
print("--- XGBoost ---")

def xgb_objective(trial):
    
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 600),
        "max_depth":        trial.suggest_int("max_depth", 3, 9),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log = True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log = True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),

        "scale_pos_weight": scale_pos_weight,
        "tree_method":      "hist",
        "random_state":     42,
        "n_jobs":           -1,
        "verbosity":        0,
    }

    scores = cross_val_score(XGBClassifier(**params), X, y, cv = 5, scoring = "roc_auc")
    return scores.mean()

xgb_study = optuna.create_study(direction = "maximize", sampler = TPESampler(seed = 42))
xgb_study.optimize(xgb_objective, n_trials = 20, show_progress_bar = True)

xgb_model = XGBClassifier(**xgb_study.best_params, scale_pos_weight = scale_pos_weight, tree_method = "hist", random_state = 42, n_jobs = -1, verbosity = 0,)
xgb_model.fit(X, y)

print(f"Best ROC AUC : {xgb_study.best_value:.4f}")
print(f"Best Params  : {xgb_study.best_params}")

--- XGBoost ---


  0%|          | 0/20 [00:00<?, ?it/s]

Best ROC AUC : 0.9087
Best Params  : {'n_estimators': 237, 'max_depth': 5, 'learning_rate': 0.06689802001564377, 'subsample': 0.5454882143733981, 'colsample_bytree': 0.8705765955899738, 'reg_alpha': 0.00265229484319886, 'reg_lambda': 2.0577776761646817e-08, 'min_child_weight': 8}


In [16]:
results["XGBoost"]         = xgb_study.best_value
best_estimators["XGBoost"] = xgb_model

In [17]:
print("--- LightGBM ---")

def lgbm_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
        "max_depth":         trial.suggest_int("max_depth", 3, 9),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 200),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log = True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log = True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),

        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs":       -1,
        "verbose":      -1,
    }
    
    scores = cross_val_score(LGBMClassifier(**params), X, y, cv = 5, scoring="roc_auc")
    return scores.mean()

lgbm_study = optuna.create_study(direction = "maximize", sampler = TPESampler(seed = 42))
lgbm_study.optimize(lgbm_objective, n_trials = 20, show_progress_bar = True)

lgbm_model = LGBMClassifier(**lgbm_study.best_params, class_weight = "balanced", random_state = 42, n_jobs = -1, verbose = -1)
lgbm_model.fit(X, y)

print(f"Best ROC AUC : {lgbm_study.best_value:.4f}")
print(f"Best Params  : {lgbm_study.best_params}")

--- LightGBM ---


  0%|          | 0/20 [00:00<?, ?it/s]

Best ROC AUC : 0.9088
Best Params  : {'n_estimators': 525, 'max_depth': 5, 'learning_rate': 0.03986232869983601, 'num_leaves': 24, 'subsample': 0.7171781358782056, 'colsample_bytree': 0.7409481648316648, 'reg_alpha': 0.0037653511867646963, 'reg_lambda': 0.012098055308424125, 'min_child_samples': 7}


In [18]:
results["LightGBM"]         = lgbm_study.best_value
best_estimators["LightGBM"] = lgbm_model

In [19]:
print("--- MLP Classifier ---")

mlp_pipe = Pipeline([("scaler", StandardScaler()),("clf", MLPClassifier(max_iter = 500, early_stopping = True, n_iter_no_change = 15, random_state = 42))])

mlp_param = {
    "clf__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64), (128, 64, 32)],
    "clf__activation":         ["relu", "tanh"],
    "clf__alpha":              uniform(1e-5, 0.1),
    "clf__learning_rate_init": uniform(1e-4, 1e-2),
    "clf__batch_size":         [128, 256, 512],
}

mlp_model = RandomizedSearchCV(mlp_pipe, mlp_param, n_iter = 10, cv = 5, scoring = "roc_auc", n_jobs = -1, random_state = 42, verbose = 1)
mlp_model.fit(X, y)

print(f"Best ROC AUC : {mlp_model.best_score_:.4f}")
print(f"Best Params  : {mlp_model.best_params_}")

--- MLP Classifier ---
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best ROC AUC : 0.9081
Best Params  : {'clf__activation': 'relu', 'clf__alpha': np.float64(0.015611864044243652), 'clf__batch_size': 512, 'clf__hidden_layer_sizes': (64, 32), 'clf__learning_rate_init': np.float64(0.004692488919658672)}


In [20]:
results["MLP"]         = mlp_model.best_score_
best_estimators["MLP"] = mlp_model.best_estimator_

In [21]:
print("--- Results Summary ---")

results = (pd.DataFrame.from_dict(results, orient = "index", columns = ["ROC_AUC"]).sort_values("ROC_AUC", ascending = False).reset_index().rename(columns = {"index": "Model"}))

results["ROC_AUC"] = results["ROC_AUC"].round(4)

print(results.to_string(index = False))

--- Results Summary ---
              Model  ROC_AUC
           LightGBM   0.9088
            XGBoost   0.9087
                MLP   0.9081
      Random Forest   0.9076
Logistic Regression   0.9063


In [22]:
joblib.dump(best_estimators, "all_models.pkl")

best_model_name = results.iloc[0]["Model"]
best_model      = best_estimators[best_model_name]
joblib.dump(best_model, "best_model.pkl")

print(f"Best model : {best_model_name}  (ROC AUC {results.iloc[0]['ROC_AUC']:.4f})")
print("Saved → best_model.pkl")
print("Saved → all_models.pkl")

Best model : LightGBM  (ROC AUC 0.9088)
Saved → best_model.pkl
Saved → all_models.pkl
